# 02 — P2P Process Analysis & Case-Level Model

**Project:** Purchase-to-Pay Process Mining & Exception Intelligence

## Objective

This notebook converts the raw **event-level P2P log** into a **case-level analytical model** and then uses that model to examine:

- process variants,
- cycle time,
- purchase-order modifications,
- invoice reversals,
- goods-receipt reversals,
- payment-block handling,
- repeated activities,
- resource handoffs,
- and completion behavior.

> **Analytical principle:** Event-level evidence is preserved. Case-level KPIs are calculated only after events are correctly grouped by `case:concept:name`.


## 1. Setup and source loading

The notebook supports both the local project structure and this portfolio workspace.

Expected local path:

```text
data/raw/output.csv
```


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

candidates = [
    Path("../data/raw/output.csv"),
    Path("data/raw/output.csv"),
    Path("/mnt/data/output.csv"),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place output.csv in data/raw/output.csv")

print(f"Using source: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Event rows: {len(df):,}")
print(f"Cases: {df['case:concept:name'].nunique():,}")


Using source: /mnt/data/output.csv


Event rows: 1,595,923
Cases: 251,734


## 2. Preserve source order and parse timestamps

Multiple events can share the same timestamp.  
To make event sequencing deterministic, the original row order is retained as a tie-breaker.


In [2]:
df["_source_order"] = np.arange(len(df))

df["event_timestamp"] = pd.to_datetime(
    df["time:timestamp"],
    errors="coerce",
    utc=True
)

print(f"Unparseable timestamps: {df['event_timestamp'].isna().sum():,}")
print(f"Minimum timestamp: {df['event_timestamp'].min()}")
print(f"Maximum timestamp: {df['event_timestamp'].max()}")


Unparseable timestamps: 0
Minimum timestamp: 1948-01-26 22:59:00+00:00
Maximum timestamp: 2020-04-09 21:59:00+00:00


## 3. Validate case-level attribute consistency

Fields prefixed with `case:` are expected to describe the process case, but this assumption should be tested before using `.first()` during case aggregation.

For each candidate field, we count how many cases contain more than one distinct non-null value.


In [3]:
case_attributes = [
    "case:Vendor",
    "case:Company",
    "case:Spend area text",
    "case:Sub spend area text",
    "case:Spend classification text",
    "case:Purchasing Document",
    "case:Document Type",
    "case:Item Type",
    "case:Item Category",
    "case:GR-Based Inv. Verif.",
    "case:Item",
    "case:Goods Receipt",
]

consistency_rows = []

for col in case_attributes:
    distinct_per_case = df.groupby("case:concept:name")[col].nunique(dropna=True)
    inconsistent_cases = int((distinct_per_case > 1).sum())

    consistency_rows.append({
        "field": col,
        "inconsistent_cases": inconsistent_cases,
        "inconsistent_case_pct": round(
            inconsistent_cases / df["case:concept:name"].nunique() * 100, 4
        )
    })

case_consistency = pd.DataFrame(consistency_rows)
display(case_consistency.sort_values("inconsistent_cases", ascending=False))


,field,inconsistent_cases,inconsistent_case_pct
0,case:Vendor,0,0.00
1,case:Company,0,0.00
2,case:Spend area text,0,0.00
3,case:Sub spend area text,0,0.00
4,case:Spend classification text,0,0.00
5,case:Purchasing Document,0,0.00
6,case:Document Type,0,0.00
7,case:Item Type,0,0.00
8,case:Item Category,0,0.00
9,case:GR-Based Inv. Verif.,0,0.00


## 4. Sort events within each case

The event log is sorted by:

1. case ID,
2. parsed timestamp,
3. original source-row order.

This ordering is used for process variants and resource-handoff calculations.


In [4]:
events = (
    df.sort_values(
        ["case:concept:name", "event_timestamp", "_source_order"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

print(events[["case:concept:name", "concept:name", "event_timestamp"]].head(10))


  case:concept:name                         concept:name  \
0  2000000000_00001                         SRM: Created   
1  2000000000_00001                        SRM: Complete   
2  2000000000_00001               SRM: Awaiting Approval   
3  2000000000_00001              SRM: Document Completed   
4  2000000000_00001  SRM: In Transfer to Execution Syst.   
5  2000000000_00001                         SRM: Ordered   
6  2000000000_00001          SRM: Change was Transmitted   
7  2000000000_00001           Create Purchase Order Item   
8  2000000000_00001               Vendor creates invoice   
9  2000000000_00001                 Record Goods Receipt   

            event_timestamp  
0 2018-01-02 12:53:00+00:00  
1 2018-01-02 13:53:00+00:00  
2 2018-01-02 13:53:00+00:00  
3 2018-01-02 13:53:00+00:00  
4 2018-01-02 13:53:00+00:00  
5 2018-01-02 13:53:00+00:00  
6 2018-01-02 13:53:00+00:00  
7 2018-01-02 13:53:00+00:00  
8 2018-01-02 22:59:00+00:00  
9 2018-03-06 06:44:00+00:00  


## 5. Define transparent event-rule groups

These rules are intentionally based on **explicit activity names** found in the source event log.

They are analytical classifications, not claims of fraud, non-compliance, or financial loss.


In [5]:
PO_MODIFICATION_EVENTS = {
    "Change Quantity",
    "Change Price",
    "Change Approval for Purchase Order",
    "Change Delivery Indicator",
    "Change Storage Location",
    "Change Currency",
    "Change payment term",
}

PO_INTERVENTION_EVENTS = {
    "Delete Purchase Order Item",
    "Block Purchase Order Item",
    "Reactivate Purchase Order Item",
}

INVOICE_REVERSAL_EVENT = "Cancel Invoice Receipt"
GR_REVERSAL_EVENT = "Cancel Goods Receipt"
PAYMENT_BLOCK_SET_EVENT = "Set Payment Block"
PAYMENT_BLOCK_REMOVE_EVENT = "Remove Payment Block"
CLEAR_INVOICE_EVENT = "Clear Invoice"

observed_activities = set(events["concept:name"].dropna().unique())

rules_check = pd.DataFrame({
    "rule_group": [
        "PO modification",
        "PO intervention",
        "Invoice reversal",
        "GR reversal",
        "Payment block set",
        "Payment block remove",
        "Invoice clearing"
    ],
    "configured_events": [
        len(PO_MODIFICATION_EVENTS),
        len(PO_INTERVENTION_EVENTS),
        1, 1, 1, 1, 1
    ],
    "events_found_in_source": [
        len(PO_MODIFICATION_EVENTS & observed_activities),
        len(PO_INTERVENTION_EVENTS & observed_activities),
        int(INVOICE_REVERSAL_EVENT in observed_activities),
        int(GR_REVERSAL_EVENT in observed_activities),
        int(PAYMENT_BLOCK_SET_EVENT in observed_activities),
        int(PAYMENT_BLOCK_REMOVE_EVENT in observed_activities),
        int(CLEAR_INVOICE_EVENT in observed_activities),
    ]
})

display(rules_check)


,rule_group,configured_events,events_found_in_source
0,PO modification,7,7
1,PO intervention,3,3
2,Invoice reversal,1,1
3,GR reversal,1,1
4,Payment block set,1,1
5,Payment block remove,1,1
6,Invoice clearing,1,1


## 6. Event-level derived fields

We derive flags at event level before aggregating to cases.

### Resource handoff rule

A handoff occurs when the resource on the current event differs from the resource on the immediately preceding event within the same case.

The first event of a case cannot be a handoff.


In [6]:
events["previous_resource"] = (
    events.groupby("case:concept:name")["org:resource"].shift(1)
)

events["resource_handoff"] = (
    events["previous_resource"].notna()
    & events["org:resource"].notna()
    & events["org:resource"].ne(events["previous_resource"])
).astype("int8")

events["po_modification_event"] = (
    events["concept:name"].isin(PO_MODIFICATION_EVENTS)
).astype("int8")

events["po_intervention_event"] = (
    events["concept:name"].isin(PO_INTERVENTION_EVENTS)
).astype("int8")

events["invoice_reversal_event"] = (
    events["concept:name"].eq(INVOICE_REVERSAL_EVENT)
).astype("int8")

events["gr_reversal_event"] = (
    events["concept:name"].eq(GR_REVERSAL_EVENT)
).astype("int8")

events["payment_block_set_event"] = (
    events["concept:name"].eq(PAYMENT_BLOCK_SET_EVENT)
).astype("int8")

events["payment_block_remove_event"] = (
    events["concept:name"].eq(PAYMENT_BLOCK_REMOVE_EVENT)
).astype("int8")

events["invoice_clear_event"] = (
    events["concept:name"].eq(CLEAR_INVOICE_EVENT)
).astype("int8")


## 7. Build the case-level analytical model

One row is created for each unique `case:concept:name`.

The model combines:

- case attributes,
- start/end timestamps,
- cycle time,
- event/activity/resource counts,
- resource handoffs,
- and exception/event flags.


In [7]:
case_id = "case:concept:name"

case_base = (
    events.groupby(case_id)
    .agg(
        vendor=("case:Vendor", "first"),
        company=("case:Company", "first"),
        spend_area=("case:Spend area text", "first"),
        sub_spend_area=("case:Sub spend area text", "first"),
        spend_classification=("case:Spend classification text", "first"),
        purchasing_document=("case:Purchasing Document", "first"),
        document_type=("case:Document Type", "first"),
        item_type=("case:Item Type", "first"),
        item_category=("case:Item Category", "first"),
        gr_based_invoice_verification=("case:GR-Based Inv. Verif.", "first"),
        goods_receipt=("case:Goods Receipt", "first"),
        case_start=("event_timestamp", "min"),
        case_end=("event_timestamp", "max"),
        event_count=("concept:name", "size"),
        unique_activity_count=("concept:name", "nunique"),
        unique_resource_count=("org:resource", "nunique"),
        resource_handoff_count=("resource_handoff", "sum"),
        po_modification_count=("po_modification_event", "sum"),
        po_intervention_count=("po_intervention_event", "sum"),
        invoice_reversal_count=("invoice_reversal_event", "sum"),
        gr_reversal_count=("gr_reversal_event", "sum"),
        payment_block_set_count=("payment_block_set_event", "sum"),
        payment_block_remove_count=("payment_block_remove_event", "sum"),
        invoice_clear_count=("invoice_clear_event", "sum"),
    )
)

case_base["cycle_time_days_raw"] = (
    (case_base["case_end"] - case_base["case_start"]).dt.total_seconds()
    / 86400
)

case_base["repeated_activity_count"] = (
    case_base["event_count"] - case_base["unique_activity_count"]
)

case_base["po_modification_flag"] = (case_base["po_modification_count"] > 0).astype("int8")
case_base["po_intervention_flag"] = (case_base["po_intervention_count"] > 0).astype("int8")
case_base["invoice_reversal_flag"] = (case_base["invoice_reversal_count"] > 0).astype("int8")
case_base["gr_reversal_flag"] = (case_base["gr_reversal_count"] > 0).astype("int8")
case_base["payment_block_set_flag"] = (case_base["payment_block_set_count"] > 0).astype("int8")
case_base["payment_block_remove_flag"] = (case_base["payment_block_remove_count"] > 0).astype("int8")
case_base["payment_block_handling_flag"] = (
    (case_base["payment_block_set_count"] > 0)
    | (case_base["payment_block_remove_count"] > 0)
).astype("int8")
case_base["invoice_cleared_flag"] = (case_base["invoice_clear_count"] > 0).astype("int8")
case_base["repeated_activity_flag"] = (case_base["repeated_activity_count"] > 0).astype("int8")

case_base = case_base.reset_index()

print(f"Case-level rows: {len(case_base):,}")
display(case_base.head())


Case-level rows: 251,734


,case:concept:name,vendor,company,spend_area,sub_spend_area,spend_classification,purchasing_document,document_type,item_type,item_category,gr_based_invoice_verification,goods_receipt,case_start,case_end,event_count,unique_activity_count,unique_resource_count,resource_handoff_count,po_modification_count,po_intervention_count,invoice_reversal_count,gr_reversal_count,payment_block_set_count,payment_block_remove_count,invoice_clear_count,cycle_time_days_raw,repeated_activity_count,po_modification_flag,po_intervention_flag,invoice_reversal_flag,gr_reversal_flag,payment_block_set_flag,payment_block_remove_flag,payment_block_handling_flag,invoice_cleared_flag,repeated_activity_flag
0,2000000000_00001,vendorID_0000,companyID_0000,CAPEX & SOCS,Facility Management,NPR,2000000000,EC Purchase order,Standard,"3-way match, invoice before GR",False,True,2018-01-02 12:53:00+00:00,2018-03-29 13:06:00+00:00,12,12,5,5,0,0,0,0,0,0,1,86.01,0,0,0,0,0,0,0,0,1,0
1,2000000001_00001,vendorID_0001,companyID_0000,Marketing,Marketing Support Services,NPR,2000000001,EC Purchase order,Service,"3-way match, invoice after GR",True,True,2018-01-03 08:49:00+00:00,2019-01-17 10:58:00+00:00,15,14,6,7,0,0,0,0,0,0,1,379.09,1,0,0,0,0,0,0,0,1,1
2,2000000002_00001,vendorID_0002,companyID_0000,Marketing,Digital Marketing,NPR,2000000002,EC Purchase order,Service,"3-way match, invoice after GR",True,True,2018-01-04 13:17:00+00:00,2019-01-17 10:59:00+00:00,18,15,5,7,0,0,0,1,0,0,1,377.90,3,0,0,0,1,0,0,0,1,1
3,2000000003_00001,vendorID_0003,companyID_0000,Enterprise Services,Office Supplies,NPR,2000000003,EC Purchase order,Standard,"3-way match, invoice before GR",False,True,2018-01-08 06:21:00+00:00,2018-03-08 12:21:00+00:00,12,12,5,5,0,0,0,0,0,0,1,59.25,0,0,0,0,0,0,0,0,1,0
4,2000000003_00002,vendorID_0003,companyID_0000,Enterprise Services,Office Supplies,NPR,2000000003,EC Purchase order,Standard,"3-way match, invoice before GR",False,True,2018-01-08 06:21:00+00:00,2018-03-08 12:21:00+00:00,12,12,5,5,0,0,0,0,0,0,1,59.25,0,0,0,0,0,0,0,0,1,0


## 8. Build process variants

A process variant is the ordered sequence of activities observed within a case.

To keep the model readable:

- the full activity sequence is retained,
- identical sequences receive the same `variant_id`,
- and variant frequency/share is calculated at case level.


In [8]:
variant_sequence = (
    events.groupby(case_id)["concept:name"]
    .agg(tuple)
    .rename("activity_sequence")
)

variant_codes, unique_variants = pd.factorize(variant_sequence, sort=False)

variant_map = pd.DataFrame({
    case_id: variant_sequence.index,
    "activity_sequence": variant_sequence.values,
    "variant_id": variant_codes + 1
})

variant_frequency = (
    variant_map.groupby("variant_id")
    .size()
    .rename("variant_case_count")
    .reset_index()
)

variant_frequency["variant_case_share_pct"] = (
    variant_frequency["variant_case_count"]
    / len(case_base)
    * 100
).round(4)

variant_map = variant_map.merge(variant_frequency, on="variant_id", how="left")

case_model = case_base.merge(
    variant_map,
    on=case_id,
    how="left"
)

print(f"Unique process variants: {case_model['variant_id'].nunique():,}")


Unique process variants: 11,973


## 9. Timestamp-anomaly flag

The raw event log contains unusually old dates.  
Rather than deleting them, the case model explicitly flags cases containing events before 2016.

This flag is used to prevent anomalous historical timestamps from silently distorting cycle-time interpretation.

> The cutoff is an analytical review flag, not a claim that those records are invalid.


In [9]:
anomalous_timestamp_cases = (
    events.loc[
        events["event_timestamp"].dt.year < 2016,
        case_id
    ]
    .dropna()
    .unique()
)

case_model["timestamp_anomaly_flag"] = (
    case_model[case_id].isin(anomalous_timestamp_cases)
).astype("int8")

print(f"Cases flagged for historical timestamps: {case_model['timestamp_anomaly_flag'].sum():,}")


Cases flagged for historical timestamps: 78


## 10. Case-model validation

The number of case-level rows must equal the number of unique source case IDs.


In [10]:
source_cases = df[case_id].nunique()
model_cases = case_model[case_id].nunique()

validation = pd.DataFrame({
    "check": [
        "Source unique cases",
        "Case-model rows",
        "Case-model unique cases",
        "Duplicate case IDs in model"
    ],
    "value": [
        source_cases,
        len(case_model),
        model_cases,
        case_model[case_id].duplicated().sum()
    ]
})

display(validation)

assert len(case_model) == source_cases
assert model_cases == source_cases
assert case_model[case_id].duplicated().sum() == 0


,check,value
0,Source unique cases,251734
1,Case-model rows,251734
2,Case-model unique cases,251734
3,Duplicate case IDs in model,0


## 11. Overall process-performance summary

Cycle-time statistics are shown both:

- for all cases,
- and for cases not flagged for unusually old timestamps.

This keeps the raw evidence while making distortion visible.


In [11]:
def cycle_summary(series):
    return series.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )

print("RAW CYCLE TIME — ALL CASES")
display(cycle_summary(case_model["cycle_time_days_raw"]))

print("\nCYCLE TIME — CASES WITHOUT HISTORICAL TIMESTAMP FLAG")
display(
    cycle_summary(
        case_model.loc[
            case_model["timestamp_anomaly_flag"].eq(0),
            "cycle_time_days_raw"
        ]
    )
)


RAW CYCLE TIME — ALL CASES


count   251,734.00
mean         71.52
std         152.78
min           0.00
50%          64.04
75%          98.26
90%         126.10
95%         142.30
99%         219.50
max      25,670.55
Name: cycle_time_days_raw, dtype: float64


CYCLE TIME — CASES WITHOUT HISTORICAL TIMESTAMP FLAG


count   251,656.00
mean         69.60
std          47.87
min           0.00
50%          64.04
75%          98.25
90%         126.08
95%         142.16
99%         217.76
max         818.96
Name: cycle_time_days_raw, dtype: float64

## 12. Exception and complexity prevalence

Rates are calculated using **unique cases as the denominator**, not event rows.


In [12]:
flag_columns = {
    "PO modification": "po_modification_flag",
    "PO intervention": "po_intervention_flag",
    "Invoice reversal": "invoice_reversal_flag",
    "GR reversal": "gr_reversal_flag",
    "Payment block set": "payment_block_set_flag",
    "Payment block remove": "payment_block_remove_flag",
    "Payment block handling": "payment_block_handling_flag",
    "Repeated activity": "repeated_activity_flag",
    "Invoice cleared": "invoice_cleared_flag",
    "Timestamp anomaly": "timestamp_anomaly_flag",
}

summary_rows = []

for label, col in flag_columns.items():
    cases = int(case_model[col].sum())
    summary_rows.append({
        "metric": label,
        "case_count": cases,
        "case_rate_pct": round(cases / len(case_model) * 100, 2)
    })

exception_summary = pd.DataFrame(summary_rows)
display(exception_summary)


,metric,case_count,case_rate_pct
0,PO modification,31675,12.58
1,PO intervention,9202,3.66
2,Invoice reversal,6471,2.57
3,GR reversal,2470,0.98
4,Payment block set,122,0.05
5,Payment block remove,55839,22.18
6,Payment block handling,55934,22.22
7,Repeated activity,22438,8.91
8,Invoice cleared,183677,72.96
9,Timestamp anomaly,78,0.03


## 13. Top process variants

For readability, the full sequence is converted to text only for the top variants.


In [13]:
top_variants = (
    case_model[
        ["variant_id", "variant_case_count", "variant_case_share_pct", "activity_sequence"]
    ]
    .drop_duplicates("variant_id")
    .sort_values("variant_case_count", ascending=False)
    .head(15)
    .copy()
)

top_variants["process_path"] = top_variants["activity_sequence"].apply(
    lambda seq: " → ".join(seq)
)

display(
    top_variants[
        ["variant_id", "variant_case_count", "variant_case_share_pct", "process_path"]
    ]
)


,variant_id,variant_case_count,variant_case_share_pct,process_path
1450,389,50286,19.98,Create Purchase Order Item → Vendor creates in...
1445,387,30798,12.23,Create Purchase Order Item → Record Goods Rece...
1636,417,12214,4.85,Create Purchase Order Item → Record Goods Receipt
1457,393,11383,4.52,Create Purchase Order Item → Vendor creates in...
1443,385,9694,3.85,Create Purchase Order Item → Receive Order Con...
2537,563,8921,3.54,Create Purchase Requisition Item → Create Purc...
1467,396,8835,3.51,Create Purchase Order Item → Vendor creates in...
1740,426,7985,3.17,Create Purchase Order Item → Record Goods Rece...
1743,427,5298,2.10,Create Purchase Order Item → Delete Purchase O...
1444,386,4244,1.69,Create Purchase Order Item → Receive Order Con...


## 14. Cycle time by major process variant

To avoid conclusions driven by tiny samples, this view focuses on variants with at least 100 cases and excludes cases carrying the historical timestamp flag from the cycle-time calculation.


In [14]:
variant_performance = (
    case_model.loc[case_model["timestamp_anomaly_flag"].eq(0)]
    .groupby("variant_id")
    .agg(
        cases=(case_id, "size"),
        median_cycle_days=("cycle_time_days_raw", "median"),
        p90_cycle_days=("cycle_time_days_raw", lambda s: s.quantile(0.90)),
        po_modification_rate=("po_modification_flag", "mean"),
        invoice_reversal_rate=("invoice_reversal_flag", "mean"),
        gr_reversal_rate=("gr_reversal_flag", "mean"),
        median_handoffs=("resource_handoff_count", "median"),
    )
    .reset_index()
)

variant_performance = variant_performance[
    variant_performance["cases"] >= 100
].copy()

for col in ["po_modification_rate", "invoice_reversal_rate", "gr_reversal_rate"]:
    variant_performance[col] = (variant_performance[col] * 100).round(2)

display(
    variant_performance
    .sort_values("median_cycle_days", ascending=False)
    .head(20)
)


,variant_id,cases,median_cycle_days,p90_cycle_days,po_modification_rate,invoice_reversal_rate,gr_reversal_rate,median_handoffs
660,664,189,196.38,218.77,0.00,100.00,100.00,2.00
1854,1860,131,194.92,252.20,0.00,0.00,0.00,3.00
405,408,1107,187.41,214.56,0.00,0.00,0.00,1.00
3702,3712,101,184.84,184.87,100.00,0.00,0.00,2.00
4707,4721,213,183.13,183.13,0.00,100.00,100.00,3.00
685,689,327,134.08,192.07,100.00,0.00,0.00,6.00
938,942,110,125.17,156.31,100.00,0.00,0.00,6.00
9022,9045,205,120.04,120.04,100.00,0.00,0.00,6.00
5860,5875,245,119.12,136.01,100.00,0.00,0.00,3.00
408,411,128,117.27,169.86,0.00,0.00,0.00,5.00


## 15. Process category comparison

`case:Item Category` distinguishes the principal matching/process groups in this dataset.

We compare case volume, cycle time, PO modification, reversals, and payment-block handling across those groups.


In [15]:
category_performance = (
    case_model.loc[case_model["timestamp_anomaly_flag"].eq(0)]
    .groupby("item_category", dropna=False)
    .agg(
        cases=(case_id, "size"),
        median_cycle_days=("cycle_time_days_raw", "median"),
        p90_cycle_days=("cycle_time_days_raw", lambda s: s.quantile(0.90)),
        po_modification_rate=("po_modification_flag", "mean"),
        invoice_reversal_rate=("invoice_reversal_flag", "mean"),
        gr_reversal_rate=("gr_reversal_flag", "mean"),
        payment_block_handling_rate=("payment_block_handling_flag", "mean"),
        median_handoffs=("resource_handoff_count", "median"),
        median_events=("event_count", "median"),
    )
    .reset_index()
)

rate_cols = [
    "po_modification_rate",
    "invoice_reversal_rate",
    "gr_reversal_rate",
    "payment_block_handling_rate"
]

for col in rate_cols:
    category_performance[col] = (category_performance[col] * 100).round(2)

display(category_performance.sort_values("cases", ascending=False))


,item_category,cases,median_cycle_days,p90_cycle_days,po_modification_rate,invoice_reversal_rate,gr_reversal_rate,payment_block_handling_rate,median_handoffs,median_events
2,"3-way match, invoice before GR",220949,66.35,127.01,12.31,2.66,0.83,24.27,4.00,5.00
1,"3-way match, invoice after GR",15165,63.39,133.18,13.14,3.69,2.90,15.07,4.00,6.00
3,Consignment,14498,19.93,48.68,9.84,0.00,1.42,0.00,1.00,2.00
0,2-way match,1044,23.69,196.86,100.00,0.96,0.00,1.15,3.00,4.00


## 16. PO modification cases vs non-modification cases

This is an **association comparison**, not a causal test.

A longer median cycle time among modified cases does not prove that the modification caused the delay.


In [16]:
po_cycle_comparison = (
    case_model.loc[case_model["timestamp_anomaly_flag"].eq(0)]
    .groupby("po_modification_flag")
    .agg(
        cases=(case_id, "size"),
        median_cycle_days=("cycle_time_days_raw", "median"),
        p90_cycle_days=("cycle_time_days_raw", lambda s: s.quantile(0.90)),
        median_events=("event_count", "median"),
        median_handoffs=("resource_handoff_count", "median"),
    )
    .reset_index()
)

po_cycle_comparison["group"] = po_cycle_comparison["po_modification_flag"].map({
    0: "No PO modification",
    1: "PO modification present"
})

display(
    po_cycle_comparison[
        ["group", "cases", "median_cycle_days", "p90_cycle_days",
         "median_events", "median_handoffs"]
    ]
)


,group,cases,median_cycle_days,p90_cycle_days,median_events,median_handoffs
0,No PO modification,220000,62.36,121.90,5.00,4.00
1,PO modification present,31656,83.00,157.01,7.00,5.00


## 17. Resource handoffs and process complexity

We summarize the relationship between resource handoffs and cycle time using handoff bands.

This is exploratory evidence only; it does not establish causation.


In [17]:
handoff_bins = [-1, 0, 2, 5, 10, np.inf]
handoff_labels = ["0", "1–2", "3–5", "6–10", "11+"]

case_model["handoff_band"] = pd.cut(
    case_model["resource_handoff_count"],
    bins=handoff_bins,
    labels=handoff_labels
)

handoff_summary = (
    case_model.loc[case_model["timestamp_anomaly_flag"].eq(0)]
    .groupby("handoff_band", observed=True)
    .agg(
        cases=(case_id, "size"),
        median_cycle_days=("cycle_time_days_raw", "median"),
        p90_cycle_days=("cycle_time_days_raw", lambda s: s.quantile(0.90)),
        median_events=("event_count", "median"),
    )
    .reset_index()
)

display(handoff_summary)


,handoff_band,cases,median_cycle_days,p90_cycle_days,median_events
0,0,9277,0.05,16.20,2.00
1,1–2,28039,17.25,66.21,2.00
2,3–5,184987,69.09,124.98,5.00
3,6–10,27060,85.72,168.20,8.00
4,11+,2293,92.52,330.27,41.00


## 18. Save the analytical case model locally

The derived case-level file is generated for downstream SQL, Excel, and Power BI work.

It is stored under `data/processed/`, which should remain excluded from GitHub because it is generated data.


In [18]:
preferred_output = Path("../data/processed/p2p_case_level.csv")
fallback_output = Path("/mnt/data/p2p_case_level.csv")

try:
    preferred_output.parent.mkdir(parents=True, exist_ok=True)
    case_output = preferred_output
except PermissionError:
    case_output = fallback_output

export_cols = [
    case_id,
    "vendor",
    "company",
    "spend_area",
    "sub_spend_area",
    "spend_classification",
    "purchasing_document",
    "document_type",
    "item_type",
    "item_category",
    "gr_based_invoice_verification",
    "goods_receipt",
    "case_start",
    "case_end",
    "cycle_time_days_raw",
    "event_count",
    "unique_activity_count",
    "unique_resource_count",
    "resource_handoff_count",
    "repeated_activity_count",
    "po_modification_count",
    "po_modification_flag",
    "po_intervention_count",
    "po_intervention_flag",
    "invoice_reversal_count",
    "invoice_reversal_flag",
    "gr_reversal_count",
    "gr_reversal_flag",
    "payment_block_set_count",
    "payment_block_set_flag",
    "payment_block_remove_count",
    "payment_block_remove_flag",
    "payment_block_handling_flag",
    "invoice_clear_count",
    "invoice_cleared_flag",
    "variant_id",
    "variant_case_count",
    "variant_case_share_pct",
    "timestamp_anomaly_flag",
]

case_model[export_cols].to_csv(case_output, index=False)

print(f"Saved case-level model: {case_output}")
print(f"Rows: {len(case_model):,}")
print(f"Columns exported: {len(export_cols)}")

Saved case-level model: /mnt/data/p2p_case_level.csv
Rows: 251,734
Columns exported: 39


## 19. Process-analysis conclusions

This notebook establishes the reusable analytical backbone of the project:

- one row per P2P case,
- deterministic event sequencing,
- cycle-time calculation,
- process-variant assignment,
- transparent exception flags,
- resource-handoff metrics,
- repeated-activity metrics,
- and process-category comparison.

### Analytical boundaries

The model does **not** interpret:

- exceptions as fraud,
- rare variants as non-compliance,
- repeated events as confirmed rework,
- vendor exception rates as supplier quality,
- or observed associations as causation.

The next notebook, **`03_root_cause_analysis.ipynb`**, will use this case-level model to investigate where process complexity and exception patterns are concentrated across vendors, spend areas, document types, and other business dimensions.
